In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
import pandas as pd
import numpy as np

In [2]:
from torchvision import datasets
from torch.utils.data import DataLoader

In [3]:
transform = transforms.Compose([
    transforms.Resize((227, 227)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5, 0.5, 0.5))
])

image_datasets = datasets.ImageFolder(
    "../Datasets/archive",
    transform = transform
)

In [4]:
from torch.utils.data import random_split

In [5]:
train_full = datasets.ImageFolder(
    "../Datasets/archive/seg_train",
    transform = transform
)

test_dataset = datasets.ImageFolder(
    "../Datasets/archive/seg_test",
    transform=transform
)

train_size = int(0.8*len(train_full))
val_size = len(train_full) - train_size

train_dataset, val_dataset = random_split(
    train_full,
    [train_size, val_size],
    generator = torch.Generator().manual_seed(42)
)

In [6]:
print("Size of Training Dataset: ",len(train_dataset))
print("Size of the Testing Dataset: ",len(test_dataset))
print("Size of the Validation Dataset: ",len(val_dataset))
print("Number of classes in the Training data are: ",(train_dataset.dataset.classes))
print("Number of classes in the Testing dataset: ",(test_dataset.classes))
print("Number of classes in the Validation dataset: ",(val_dataset.dataset.classes))

Size of Training Dataset:  11227
Size of the Testing Dataset:  3000
Size of the Validation Dataset:  2807
Number of classes in the Training data are:  ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Number of classes in the Testing dataset:  ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Number of classes in the Validation dataset:  ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [7]:
trainLoader = DataLoader(train_dataset, batch_size =128, shuffle=True)
valLoader = DataLoader(val_dataset, batch_size = 128)
testLoader = DataLoader(test_dataset, batch_size =128)

In [8]:
# CNN Architecture
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(

            nn.Conv2d(3, 96, kernel_size = 11, stride =4),
            nn.ReLU(),
            nn.MaxPool2d(3,2),

            nn.Conv2d(96, 256, kernel_size =5, stride =1, padding =2),
            nn.ReLU(),
            nn.MaxPool2d(3,2),

            nn.Conv2d(256, 384, kernel_size=3, stride =1, padding=1),
            nn.ReLU(),

            nn.Conv2d(384, 384,kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(3,2)

        )

        self.full_layers = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(6*6*256,4096 ),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Linear(4096, 6)
        )


    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.full_layers(x)
        return x

In [9]:
import torch.optim as optim
model = CNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [10]:
# Training CNN
epochs = 10
best_val_loss = float("inf")
for epoch in range(epochs):
    model.train()
    train_running_loss = 0.0
    for images, labels in trainLoader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_running_loss +=loss.item()

    epoch_train_loss = train_running_loss / len(trainLoader)

    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for images, labels in valLoader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item()

        epoch_val_loss = val_running_loss / len(valLoader)

    print(f"Epoch {epoch+1}/{epochs} Training Loss = {epoch_train_loss} & Validation Loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss :
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "Intel.pt") 

Epoch 1/10 Training Loss = 1.4776230054822834 & Validation Loss = 1.0464795367284254
Epoch 2/10 Training Loss = 1.0217111246152357 & Validation Loss = 0.9279215173287825
Epoch 3/10 Training Loss = 0.8880109942772172 & Validation Loss = 0.8429335870526053
Epoch 4/10 Training Loss = 0.7815658937801014 & Validation Loss = 0.7208125022324648
Epoch 5/10 Training Loss = 0.7276381118731066 & Validation Loss = 0.6315540942278776
Epoch 6/10 Training Loss = 0.6290573728355494 & Validation Loss = 0.5728861527009443
Epoch 7/10 Training Loss = 0.6096782213584944 & Validation Loss = 0.5761866081844677
Epoch 8/10 Training Loss = 0.5604566156186841 & Validation Loss = 0.5403687547553669
Epoch 9/10 Training Loss = 0.5160742645913904 & Validation Loss = 0.5463221818208694
Epoch 10/10 Training Loss = 0.5032242771915414 & Validation Loss = 0.516631689938632


In [11]:
# Accuracy Calculation on the Testing Data
correct_labels = 0
total_labels = 0
model.eval()
with torch.no_grad():
    for images, labels in testLoader:
        outputs = model(images)
        _, prediction = torch.max(outputs, dim=1)
                                                                                                                                                                                                                                                            
        correct_labels += (prediction == labels).sum().item()
        total_labels += labels.size(0)

print("Accuracy: ", (correct_labels / total_labels)*100)

Accuracy:  82.03333333333333
